# Refined Stage 3G validating the RQ4 explanation

**No GPU.** Runs on `refined_stage2_results.csv` plus the cached panel.

RQ4 asks *to what extent* differences in zero-shot performance can be explained by
measurable characteristics. Stage 3A answered that with correlations. Three things make
those correlations weaker evidence than they appear, and this notebook addresses each.

| Problem in 3A | Why it matters | What 3G does |
|---|---|---|
| The 132 rows are 5 series x 3 granularities x 3 models — each series contributes 9 | The rows are not independent, so p-values are optimistic | Per-model analysis, plus standard errors clustered by series |
| Only 4 descriptors, one of which (seasonal strength) was not significant | "Measurable characteristics" is a thin feature set | 9 descriptors, all computable before any model runs |
| A correlation is not an explanation test | RQ4 claims the pattern is *explained*; the contribution claim is that failure is predictable *in advance* | Leave-one-energy-type-out: fit on two types, predict the third |

The third is the important one. If a rule fitted on demand and solar predicts wind — a
signal it has never seen — that is a genuine claim about deployability. A within-sample
correlation is not.

In [11]:
import sys, os, warnings, importlib
warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import refined_stage_common as C
importlib.reload(C)
assert "extended_descriptors" in getattr(C, "__features__", set()), (
    "refined_stage_common is missing the 3G additions — restart the kernel and Run All")
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 40)
print(f"common module {C.__version__} from {C.__file__}")

common module 2026.09.10-relmae from e:\Thesis\EnergyForcastModel\refined_stage_common.py


## 1. Descriptors

Nine, all interpretable and all computable from the raw series alone.

In [12]:
from scipy import signal as sps

def acf_at(x, lag):
    if len(x) <= lag + 10: return np.nan
    a, b = x[:-lag], x[lag:]
    if np.std(a) < 1e-9 or np.std(b) < 1e-9: return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def spectral_entropy(x):
    """Shannon entropy of the normalised power spectrum, 0 = pure tone, 1 = white noise.
    Low entropy means concentrated periodic structure, which should be learnable."""
    if len(x) < 64 or np.std(x) < 1e-9: return np.nan
    _, p = sps.welch(x - x.mean(), nperseg=min(1024, len(x)))
    p = p[p > 0]
    if p.size == 0: return np.nan
    p = p / p.sum()
    return float(-(p * np.log(p)).sum() / np.log(p.size))

def trend_strength(x, m):
    """Share of variance carried by the slow (>= one seasonal period) component.

    NOT the STL definition — a centred rolling mean of width m is used instead, because
    STL on 527,040 one-minute points with period 1440 is impractical. Report it as
    'slow-component variance share', not as STL trend strength.
    """
    m = int(max(2, min(m, len(x) // 4)))
    s = pd.Series(x)
    slow = s.rolling(m, center=True, min_periods=m // 2).mean()
    resid = s - slow
    v_all, v_res = np.nanvar(s), np.nanvar(resid)
    if v_all < 1e-12: return np.nan
    return float(np.clip(1 - v_res / v_all, 0, 1))

def intermittency(x):
    """Croston-style pair. ADI = mean gap between non-zero observations (1 = never zero);
    CV2 = squared coefficient of variation of the non-zero values."""
    nz = np.flatnonzero(x != 0)
    if nz.size < 2: return np.nan, np.nan
    adi = float(np.mean(np.diff(nz)))
    v = x[nz]
    cv2 = float((np.std(v) / np.mean(v)) ** 2) if np.mean(v) > 1e-12 else np.nan
    return adi, cv2

def profile(v, m):
    mean = float(np.mean(v))
    adi, cv2 = intermittency(v)
    return {"n_points": len(v), "m": int(m),
            "pct_zero": 100 * float(np.mean(v == 0)),
            "cv": float(np.std(v) / mean) if mean > 1e-9 else np.nan,
            "seasonal_strength": acf_at(v, int(m)),
            "acf1": acf_at(v, 1),
            "spectral_entropy": spectral_entropy(v),
            "trend_strength": trend_strength(v, m),
            "adi": adi, "cv2": cv2,
            "mean": mean, "std": float(np.std(v))}

FEATURES = ["pct_zero", "cv", "seasonal_strength", "acf1",
            "spectral_entropy", "trend_strength", "adi", "cv2", "cycle_coverage"]
print(f"{len(FEATURES)} descriptors, all computable before any model runs")

9 descriptors, all computable before any model runs


In [13]:
DATA = C.get_data()
rows = []
for et in C.BASE:
    for gk in C.GRANULARITIES:
        m = C.seasonal_m(et, gk)
        for sid, v in enumerate(C.panel(DATA, et, gk)):
            rows.append({"etype": et, "gran": gk, "series_id": sid,
                         "step_minutes": C.step_minutes(et, gk), **profile(v, m)})
prof = pd.DataFrame(rows)
print(prof.groupby(["etype", "gran"])[FEATURES[:-1]].median().round(3).to_string())

loaded cached panel <- refined_panel_cache.pkl
  load   5 series, 230,736–232,272 points
  solar  5 series, 52,560–52,560 points
  wind   5 series, 434,876–434,876 points
              pct_zero     cv  seasonal_strength   acf1  spectral_entropy  trend_strength    adi    cv2
etype gran                                                                                             
load  1D         0.000  0.115              0.767  0.762             0.614           0.627  1.000  0.013
      1h         0.000  0.202              0.883  0.949             0.448           0.415  1.000  0.041
      native     0.000  0.203              0.883  0.984             0.425           0.411  1.000  0.041
solar 1D         0.000  0.377              0.248  0.424             0.789           0.397  1.000  0.142
      1h        51.804  1.458              0.853  0.927             0.394           0.070  2.072  0.498
      native    55.289  1.475              0.843  0.994             0.342           0.068  2.233  0.4

## 2. Join to accuracy

In [14]:
res = pd.read_csv("refined_stage2_results.csv")
cl = res[~res.degenerate].dropna(subset=["relMAE"])

acc = (cl.groupby(["etype", "gran", "series_id", "model"])
         .agg(relMAE=("relMAE", "median"), MASE=("MASE", "median"),
              context_steps=("context_steps", "first")).reset_index())
df = acc.merge(prof, on=["etype", "gran", "series_id"], how="left")
df["cycle_coverage"] = df.context_steps / df.m
df["log_relMAE"] = np.log10(df.relMAE.replace(0, np.nan))
df["deployable"] = (df.relMAE < 1.0).astype(int)
df = df.dropna(subset=["log_relMAE"])
print(f"{len(df)} rows = {df.series_id.nunique()} series x {df.gran.nunique()} gran "
      f"x {df.model.nunique()} models x {df.etype.nunique()} types")
print(f"deployable: {df.deployable.mean():.1%}")
df.to_csv("refined_stage3g_features.csv", index=False)

132 rows = 5 series x 3 gran x 3 models x 3 types
deployable: 56.8%


## 3. Per-model correlations

3A pooled all three models, which triples every series. Splitting by model removes that
particular dependency and shows whether a descriptor explains *all* the models or only one.

In [15]:
from scipy import stats
out = []
for mdl, g in df.groupby("model"):
    for f in FEATURES:
        ok = g[f].notna()
        if ok.sum() < 10: continue
        rho, p = stats.spearmanr(g.loc[ok, f], g.loc[ok, "log_relMAE"])
        out.append({"model": mdl, "descriptor": f, "rho": rho, "p": p, "n": int(ok.sum())})
permodel = pd.DataFrame(out)
piv = permodel.pivot(index="descriptor", columns="model", values="rho")
piv["consistent_sign"] = (np.sign(piv).nunique(axis=1) == 1)
print("Spearman rho with log relMAE, per model:\n")
print(piv.round(3).to_string())
print("\nA descriptor that flips sign between models is not explaining the phenomenon.")

Spearman rho with log relMAE, per model:

model              chronos  moirai  timesfm  consistent_sign
descriptor                                                  
acf1                 0.315   0.401    0.360             True
adi                  0.610   0.317    0.580             True
cv                   0.611   0.335    0.634             True
cv2                  0.453   0.318    0.616             True
cycle_coverage      -0.411  -0.544   -0.516             True
pct_zero             0.610   0.317    0.580             True
seasonal_strength   -0.051   0.183    0.099            False
spectral_entropy    -0.233  -0.248   -0.324             True
trend_strength      -0.230  -0.476   -0.500             True

A descriptor that flips sign between models is not explaining the phenomenon.


## 4. Standard errors clustered by series

Each series appears 9 times. Clustering by series stops those repeats from being counted
as independent evidence.

In [16]:
import statsmodels.api as sm
X = df[["pct_zero", "cv", "spectral_entropy", "trend_strength"]].copy()
X["log_cov"] = np.log10(df.cycle_coverage.clip(lower=1e-3))
ok = X.notna().all(axis=1)
X, y = sm.add_constant(X[ok]), df.loc[ok, "log_relMAE"]
groups = (df.loc[ok, "etype"] + "_" + df.loc[ok, "series_id"].astype(str))

naive = sm.OLS(y, X).fit()
clust = sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": groups})
cmp = pd.DataFrame({"coef": clust.params,
                    "p_naive": naive.pvalues, "p_clustered": clust.pvalues})
print(cmp.round(4).to_string())
print(f"\nR-squared {clust.rsquared:.3f} on {int(ok.sum())} rows, "
      f"{groups.nunique()} clusters")
print("\nCompare the two p columns: the clustered ones are the honest figures.")

                    coef  p_naive  p_clustered
const             0.0344   0.5069       0.6110
pct_zero          0.0019   0.0069       0.0744
cv                0.0217   0.3665       0.2489
spectral_entropy  0.0467   0.6168       0.5538
trend_strength   -0.1512   0.0083       0.0001
log_cov          -0.0479   0.1217       0.3590

R-squared 0.329 on 132 rows, 15 clusters

Compare the two p columns: the clustered ones are the honest figures.


## 5. The real test leave one energy type out

Fit on two energy types, predict the third. The held-out type's descriptors are available
(they need no model run), but none of its *accuracy* was seen during fitting.

Two baselines keep this honest: predicting the training mean, and a permutation of the
descriptors. Beating neither would mean the features carry nothing.

In [17]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error

FEAT = ["pct_zero", "cv", "spectral_entropy", "trend_strength", "acf1"]
d = df.dropna(subset=FEAT + ["log_relMAE"]).copy()
d["log_cov"] = np.log10(d.cycle_coverage.clip(lower=1e-3))
USE = FEAT + ["log_cov"]

rows = []
for held in d.etype.unique():
    tr, te = d[d.etype != held], d[d.etype == held]
    mdl = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(tr[USE], tr.log_relMAE)
    pred = mdl.predict(te[USE])
    base = np.full(len(te), tr.log_relMAE.mean())
    rho, p = stats.spearmanr(pred, te.log_relMAE)
    rows.append({"held_out": held, "n_train": len(tr), "n_test": len(te),
                 "MAE_model": mean_absolute_error(te.log_relMAE, pred),
                 "MAE_mean_baseline": mean_absolute_error(te.log_relMAE, base),
                 "spearman_pred_vs_actual": rho, "p": p})
loto = pd.DataFrame(rows)
loto["beats_baseline"] = loto.MAE_model < loto.MAE_mean_baseline
print(loto.round(3).to_string(index=False))
print(f"\nbeats the mean baseline on {int(loto.beats_baseline.sum())} of {len(loto)} "
      "held-out energy types")
loto.to_csv("refined_stage3g_leave_one_type_out.csv", index=False)

held_out  n_train  n_test  MAE_model  MAE_mean_baseline  spearman_pred_vs_actual     p  beats_baseline
    load       87      45      0.124              0.134                    0.315 0.035            True
   solar       87      45      0.125              0.154                    0.820 0.000            True
    wind       90      42      0.353              0.090                   -0.242 0.122           False

beats the mean baseline on 2 of 3 held-out energy types


### 5.1 The operational version, can deployability be called in advance?

Regression on a continuous score is one thing; the decision an operator actually makes is
binary. *Will a zero-shot forecast beat the naive rule on this signal?*

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

rows = []
for held in d.etype.unique():
    tr, te = d[d.etype != held], d[d.etype == held]
    if tr.deployable.nunique() < 2 or te.deployable.nunique() < 2:
        rows.append({"held_out": held, "note": "one class only — undefined"}); continue
    clf = make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=2000)).fit(tr[USE], tr.deployable)
    pr = clf.predict(te[USE]); pp = clf.predict_proba(te[USE])[:, 1]
    majority = tr.deployable.mode()[0]
    rows.append({"held_out": held, "n_test": len(te),
                 "accuracy": accuracy_score(te.deployable, pr),
                 "majority_baseline": accuracy_score(te.deployable,
                                                     np.full(len(te), majority)),
                 "roc_auc": roc_auc_score(te.deployable, pp),
                 "actual_deployable_rate": te.deployable.mean()})
clf_res = pd.DataFrame(rows)
print(clf_res.round(3).to_string(index=False))
print("\nAUC above 0.5 on a held-out energy type is the claim worth making:")
print("deployability is callable from the data before any model is run.")
clf_res.to_csv("refined_stage3g_deployability_prediction.csv", index=False)

held_out  n_test  accuracy  majority_baseline  roc_auc  actual_deployable_rate
    load      45     0.667              0.667    0.713                   0.667
   solar      45     0.600              0.533    0.881                   0.533
    wind      42     0.476              0.500    0.422                   0.500

AUC above 0.5 on a held-out energy type is the claim worth making:
deployability is callable from the data before any model is run.


## 6. RQ2 housekeeping correct the context tests

3E applied Holm to the model comparisons but not to the 84 adjacent-context tests. With 84
tests at alpha = 0.05, roughly 4 will look significant by chance alone.

In [19]:
from statsmodels.stats.multitest import multipletests
f = "refined_stage3e_context_significance.csv"
if os.path.exists(f):
    ctx = pd.read_csv(f)
    rej, padj, _, _ = multipletests(ctx.p, method="holm")
    ctx["p_holm"], ctx["significant_holm"] = padj, rej
    print(f"{len(ctx)} tests | raw significant {int(ctx.significant.sum())} "
          f"(chance alone gives ~{0.05*len(ctx):.1f}) | after Holm {int(rej.sum())}")
    if rej.sum():
        print("\nSurviving context effects:")
        print(ctx[rej][["etype", "gran", "model", "from", "to",
                        "improvement_%", "p", "p_holm"]].round(4).to_string(index=False))
        print("\nNote the sign of improvement_% — a significant DEGRADATION is not support")
        print("for 'more context helps'.")
    ctx.to_csv("refined_stage3e_context_significance_holm.csv", index=False)
else:
    print(f"{f} not found — run refined_stage3e_significance first")

84 tests | raw significant 41 (chance alone gives ~4.2) | after Holm 25

Surviving context effects:
etype   gran   model  from   to  improvement_%      p  p_holm
 load     1D chronos   256  512         9.3340 0.0000  0.0000
 load     1D  moirai   512 1024         4.6688 0.0004  0.0276
 load     1D timesfm   128  256        10.3636 0.0000  0.0000
 load     1D timesfm   256  512        27.8924 0.0000  0.0000
 load     1D timesfm   512 1024        28.5188 0.0000  0.0000
 load     1D timesfm  1024 2048        17.4807 0.0000  0.0000
 load native chronos   512 1024        14.9087 0.0001  0.0062
 load native  moirai   128  256         7.2366 0.0002  0.0162
 load native  moirai  1024 2048         1.2241 0.0000  0.0013
 load native timesfm   128  256        19.5292 0.0008  0.0472
 load native timesfm   256  512        16.1586 0.0003  0.0169
 load native timesfm  1024 2048        11.9255 0.0003  0.0207
solar     1h  moirai   128  256         9.1785 0.0007  0.0416
solar     1h  moirai   512 1024 

## 7. RQ2 housekeeping, report configuration in duration as well as steps

RQ2 asks for both observation counts and real-world duration. Every results file already
carries `history_minutes` and `lead_minutes`, so this needs no re-run.

In [20]:
for f, label in [("refined_stage3b_context_sweep.csv", "context sweep"),
                 ("refined_stage3c_horizon_sweep.csv", "horizon sweep")]:
    if not os.path.exists(f):
        print(f"{f} missing"); continue
    x = pd.read_csv(f)
    x = x[~x.degenerate].dropna(subset=["relMAE"])
    col = "context_steps" if "context" in label else "horizon_steps"
    dur = "history_minutes" if "context" in label else "lead_minutes"
    t = (x.groupby(["etype", "gran_label", col])
           .agg(duration=(dur, "first"), relMAE=("relMAE", "median")).reset_index())
    t["duration"] = t.duration.map(C.human_duration)
    print(f"\n=== {label}: steps AND duration ===")
    print(t.round(3).to_string(index=False))


=== context sweep: steps AND duration ===
etype       gran_label  context_steps duration  relMAE
 load            daily            128  128.0 d   0.851
 load            daily            256  256.0 d   0.838
 load            daily            512  512.0 d   0.720
 load            daily           1024 1024.0 d   0.642
 load            daily           2048 2048.0 d   0.605
 load           hourly            128    5.3 d   0.979
 load           hourly            256   10.7 d   0.896
 load           hourly            512   21.3 d   0.887
 load           hourly           1024   42.7 d   0.870
 load           hourly           2048   85.3 d   0.862
 load minutes (native)            128    2.7 d   1.043
 load minutes (native)            256    5.3 d   0.975
 load minutes (native)            512   10.7 d   0.858
 load minutes (native)           1024   21.3 d   0.735
 load minutes (native)           2048   42.7 d   0.719
solar           hourly            128    5.3 d   1.146
solar           hourly

## 8. How to write this up

- Quote the **clustered** p-values from section 4, not 3A's pooled ones.
- Lead RQ4 with section 5: a descriptor set fitted on two energy types that predicts the
  third is an explanation claim; a within-sample correlation is a description.
- If section 5.1's AUC is materially above 0.5, that sentence is your contribution:
  *deployability can be called from measurable properties before any model is run.*
  If it is near 0.5, say so — a negative result here is still a result, and it is the
  honest boundary of what three energy types can support.
- Section 6 tightens RQ2: report the Holm-corrected count and name the cells that survive.
- Cite TimeTic (arXiv:2509.23695) and state the difference: they learn a general
  transferability predictor across domains; this tests whether a handful of interpretable,
  domain-meaningful descriptors suffices within energy.

**Honest limits to state:** three energy types means three leave-one-out folds; load is
capped at five series because the Monash set contains only five; and the descriptors are
computed on the same period the models were evaluated on.